# 08 — Entraînement du JEPA d'écoute sur Colab (GPU)

**Prérequis (à faire AVANT) :**
1. **Runtime → Modifier le type d'exécution → GPU** (sinon AMP OFF + très lent).
2. **Le code doit être poussé sur GitHub** (Jules push — sinon Colab tourne l'ancien code).
3. **Uploader les 2 artefacts sur Drive** dans le dossier `DRIVE_DIR` ci-dessous :
   `lastfm/data/processed/sessions.parquet` (35 Mo) et `maps.pkl` (9 Mo).

Le clone ne contient PAS `lastfm/data/` (gitignoré) → d'où l'upload Drive.

In [ ]:
# --- Config (à ajuster si besoin) ---
REPO_URL = 'https://github.com/JulesV19/customer-behavior-jepa'
REPO_DIR = '/content/customer-behavior-jepa'
LASTFM   = REPO_DIR + '/lastfm'
DRIVE_DIR = '/content/drive/MyDrive/cb_jepa/lastfm_processed'   # où sont les artefacts
RUN_NAME = 'jepa_lastfm_d128'                                    # nom du checkpoint sauvé

In [ ]:
# GPU présent ?
!nvidia-smi -L
import torch; print('CUDA dispo :', torch.cuda.is_available())

In [ ]:
# Clone (ou pull si déjà cloné)
import os
if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

In [ ]:
# Monte le Drive puis copie les artefacts dans lastfm/data/processed/
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p {LASTFM}/data/processed
!cp "{DRIVE_DIR}/sessions.parquet" {LASTFM}/data/processed/
!cp "{DRIVE_DIR}/maps.pkl"         {LASTFM}/data/processed/
!ls -lh {LASTFM}/data/processed/

In [ ]:
# pyarrow au cas où — NE PAS réinstaller torch (Colab a la version CUDA)
!pip install -q pyarrow

## Entraînement
Vérifie les 2 lignes de log au démarrage : `device=cuda` et `AMP ... : ON`.
Surveille `eval séance cos` (détecteur d'effondrement) : doit rester bas (< ~0.5).

In [ ]:
import os, sys
os.chdir(LASTFM)
sys.path.insert(0, '.')
from src.train_jepa import run

hist = run(epochs=15, batch_size=128, num_workers=2)   # d=128 par défaut ; monter à d=256 pour un run costaud

In [ ]:
# Sauve le checkpoint + l'historique sur le Drive (disque Colab éphémère)
!cp {LASTFM}/data/processed/jepa.pt            "{DRIVE_DIR}/{RUN_NAME}.pt"
!cp {LASTFM}/data/processed/train_history.json "{DRIVE_DIR}/train_history_{RUN_NAME}.json"
print('checkpoint sauvé ->', DRIVE_DIR, RUN_NAME)

## Éval rapide (optionnel, juste après le run)
Retrieval morceaux + séance, contre les baselines répétition/popularité, + probe artiste.

In [ ]:
from src.lastfm_data_prep import load_all
from src.train_jepa import load_model
from src.evaluate_jepa import evaluate_model
import json

seq, maps = load_all()
model = load_model(device='cuda')
res = evaluate_model(model, seq, maps, device='cuda', eval_users=2000)
print(json.dumps({k: (v if not isinstance(v, dict) else {kk: round(vv,3) for kk,vv in v.items()})
                  for k,v in res.items()}, indent=1, ensure_ascii=False, default=lambda x: round(x,3)))